# 可选实验：Multiclass Classification


## 1.1 目标

本实验将通过示例探索使用 neural network 进行 multiclass classification（多分类）。

<figure>
  <img src="./images/C2_W2_mclass_header.png" alt="Multiclass classification 示意图" style="width:500px;max-width:100%;height:auto;">
</figure>


## 1.2 工具

本实验使用当前目录下 `lab_utils_multiclass_TF.py` 中提供的绘图函数。


In [1]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib widget
from sklearn.datasets import make_blobs
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
np.set_printoptions(precision=2)
from lab_utils_multiclass_TF import *
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

## 2.0 Multiclass Classification

Neural network 经常用于 classification，例如：

- 输入照片，将图中主体分为 dog、cat、horse、other 等类别。
- 输入句子，判断其中各词的词性，如 noun、verb、adjective 等。

这类网络的最后一层包含多个 unit，每个输出对应一个类别。输入样本后，输出值最大的类别就是预测类别。将这些输出经过 softmax，可以得到样本属于各个类别的概率。

本实验将使用 TensorFlow 构建一个 multiclass network，再观察网络如何形成预测。先创建一个包含 4 个类别的数据集。


### 2.1 准备与可视化数据

使用 scikit-learn 的 `make_blobs` 生成 4 类训练数据，如下图所示。


In [2]:
# make 4-class dataset for classification
classes = 4
m = 100
centers = [[-5, 2], [-2, -2], [1, 2], [5, -2]]
std = 1.0
X_train, y_train = make_blobs(n_samples=m, centers=centers, cluster_std=std,random_state=30)

In [5]:
plt_mc(X_train,y_train,classes, centers, std=std)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

每个点代表一个训练样本，坐标 $(x_0,x_1)$ 是输入，颜色表示所属类别。训练完成后，模型接收新样本 $(x_0,x_1)$ 并预测其类别。

虽然数据是生成的，但它代表了一类常见的实际问题：给定多个输入 feature，预测若干输出类别中的一个。


In [6]:
# show classes in data set
print(f"unique classes {np.unique(y_train)}")
# show how classes are represented
print(f"class representation {y_train[:10]}")
# show shapes of our dataset
print(f"shape of X_train: {X_train.shape}, shape of y_train: {y_train.shape}")

unique classes [0 1 2 3]
class representation [3 3 3 0 3 3 3 3 2 0]
shape of X_train: (100, 2), shape of y_train: (100,)


### 2.2 模型

<img align="right" src="./images/C2_W2_mclass_lab_network.PNG" alt="四分类网络结构" style="width:350px;padding:10px 20px;">

本实验使用图中的两层网络。与前面的 binary classification 网络不同，这里有 4 个输出，每个对应一个类别；输出值最大的类别为预测结果。

下面展示 TensorFlow 实现。注意，输出层使用 `linear` activation。虽然可以直接使用 `softmax`，但训练时将 linear 输出交给 loss function 统一处理，通常具有更好的 numerical stability。需要概率预测时，再应用 softmax 即可。


In [7]:
tf.random.set_seed(1234)  # applied to achieve consistent results
model = Sequential(
    [
        Dense(2, activation = 'relu',   name = "L1"),
        Dense(4, activation = 'linear', name = "L2")
    ]
)

下面编译并训练网络。Loss function 的参数 `from_logits=True` 表示输入来自 linear 输出层，尚未经过 softmax。


In [8]:
model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(0.01),
)

model.fit(
    X_train,y_train,
    epochs=200
)

Epoch 1/200
4/4 [==============================] - 0s 1ms/step - loss: 1.8158
Epoch 2/200
4/4 [==============================] - 0s 1ms/step - loss: 1.6976
Epoch 3/200
4/4 [==============================] - 0s 1ms/step - loss: 1.5989
Epoch 4/200
4/4 [==============================] - 0s 1ms/step - loss: 1.5179
Epoch 5/200
4/4 [==============================] - 0s 988us/step - loss: 1.4369
Epoch 6/200
4/4 [==============================] - 0s 1ms/step - loss: 1.3756
Epoch 7/200
4/4 [==============================] - 0s 1ms/step - loss: 1.3154
Epoch 8/200
4/4 [==============================] - 0s 1ms/step - loss: 1.2621
Epoch 9/200
4/4 [==============================] - 0s 1ms/step - loss: 1.2188
Epoch 10/200
4/4 [==============================] - 0s 1ms/step - loss: 1.1791
Epoch 11/200
4/4 [==============================] - 0s 1ms/step - loss: 1.1446
Epoch 12/200
4/4 [==============================] - 0s 997us/step - loss: 1.1129
Epoch 13/200
4/4 [==============================] - 0s 1m

4/4 [==============================] - 0s 1ms/step - loss: 0.1709
Epoch 105/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1662
Epoch 106/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1616
Epoch 107/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1575
Epoch 108/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1527
Epoch 109/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1480
Epoch 110/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1439
Epoch 111/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1396
Epoch 112/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1357
Epoch 113/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1315
Epoch 114/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1277
Epoch 115/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1240
Epoch 116/200
4/4 [==============================] - 0

训练完成后，下面观察模型对训练数据的分类结果。


In [9]:
plt_cat_mc(X_train, y_train, model, classes)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

上面的 decision boundary 展示了模型如何划分输入空间。这个简单模型已经成功区分训练数据，下面进一步观察其内部计算。

我们将提取训练得到的 weight，绘制每个 unit 的函数，并解释结果。使用 neural network 并不要求掌握每个 unit 的行为，但这样的观察有助于理解不同 layer 如何配合解决 classification 问题。


In [10]:
# gather the trained parameters from the first layer
l1 = model.get_layer("L1")
W1,b1 = l1.get_weights()

In [11]:
# plot the function of the first layer
plt_layer_relu(X_train, y_train.reshape(-1,), W1, b1, classes)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [12]:
# gather the trained parameters from the output layer
l2 = model.get_layer("L2")
W2, b2 = l2.get_weights()
# create the 'new features', the training examples after L1 transformation
Xl2 = np.zeros_like(X_train)
Xl2 = np.maximum(0, np.dot(X_train,W1) + b1)

plt_output_layer_linear(Xl2, y_train.reshape(-1,), W2, b2, classes,
                        x0_rng = (-0.25,np.amax(Xl2[:,0])), x1_rng = (-0.25,np.amax(Xl2[:,1])))

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

## 结果说明

### Layer 1

<img align="right" src="./images/C2_W2_mclass_layer1.png" alt="第一层两个 unit 的输出" style="width:600px;padding:10px 20px;">

图中展示第一层 unit 0 和 unit 1 的计算。坐标轴为输入 $(x_0,x_1)$，背景颜色表示输出，具体数值可参考 color bar。由于使用 ReLU，输出不局限于 0–1，本例峰值甚至超过 20。

等高线标出了 activation $a_j^{[1]}$ 从 0 变为非零的位置，也就是 ReLU 的折点。

<img src="./images/C2_W2_mclass_relu.png" alt="ReLU 的折点" style="width:200px;padding:10px 20px;">

- Unit 0 将类别 0、1 与类别 2、3 分开：直线左侧（类别 0、1）输出为 0，右侧输出大于 0。
- Unit 1 将类别 0、2 与类别 1、3 分开：直线上方（类别 0、2）输出为 0，下方输出大于 0。

下面观察这些输出如何被下一层使用。


### Layer 2：输出层

<img align="right" src="./images/C2_W2_mclass_layer2.png" alt="输出层四个 unit 的计算" style="width:600px;padding:10px 20px;">

图中的点是训练样本经过第一层变换后的表示。可以把第一层理解为创建了一组新 feature，供第二层使用。坐标轴分别为 $a_0^{[1]}$ 和 $a_1^{[1]}$。

正如前面所述，类别 0、1（蓝色、绿色）的 $a_0^{[1]}=0$，类别 0、2（蓝色、橙色）的 $a_1^{[1]}=0$。背景颜色深浅表示各个输出 unit 的数值。

- Unit 0 在接近 $(0,0)$ 的区域输出较大，对应类别 0（蓝色）。
- Unit 1 在左上角输出较大，对应类别 1（绿色）。
- Unit 2 在右下角输出较大，对应类别 2（橙色）。
- Unit 3 在右上角输出较大，对应类别 3（紫色）。

还需要注意，各 unit 的数值必须彼此协调：某个 unit 在目标类别上输出较大还不够，它必须比其他 unit 的输出都大。训练时，`SparseCategoricalCrossentropy` 内部的 softmax 将所有输出联系起来，从而实现这种协调。与逐元素的 activation 不同，softmax 同时作用于整组输出。

即使不逐一分析每个 unit，也能使用 neural network；这个例子旨在帮助你直观理解网络内部发生的计算。


## 恭喜完成实验！

你已经学会构建并使用 multiclass classification neural network。
